# Embrix Node SDK — Full Demo from Jupyter



This notebook exercises the **npm package** using a Python kernel that shells out to real `node` processes.



That means you can validate the Node package from Jupyter **without** installing a JavaScript notebook kernel.



Before running everything, make sure `embrix-node/dist` exists (it already does in this workspace after `npm run build`).



This notebook now prefers **Node 22** automatically when it is available, which avoids the native-module issues that occurred with Node 25 in this workspace.


In [4]:
from __future__ import annotations



import json

import os

import shutil

import subprocess

import sys

import textwrap

from pathlib import Path



ROOT = Path('/Users/hemantkumarthapa/Desktop/RAG/VB')

NODE_PKG = ROOT / 'embrix-node'

NODE22 = Path('/opt/homebrew/opt/node@22/bin/node')

NODE_BIN = str(NODE22 if NODE22.exists() else Path(shutil.which('node') or 'node'))



def load_dotenv(dotenv_path: Path) -> None:

    if not dotenv_path.exists():

        return

    for raw_line in dotenv_path.read_text().splitlines():

        line = raw_line.strip()

        if not line or line.startswith('#') or '=' not in line:

            continue

        key, value = line.split('=', 1)

        key = key.strip()

        value = value.strip().strip('"').strip("'")

        if key and value and key not in os.environ:

            os.environ[key] = value



def load_local_api_key(key_name: str, fallback_file: Path) -> None:

    if os.environ.get(key_name):

        return

    if fallback_file.exists():

        value = fallback_file.read_text().strip()

        if value:

            os.environ[key_name] = value



load_dotenv(ROOT / '.env')

load_local_api_key('OPENAI_API_KEY', ROOT / 'openai-api')



def require_env(name: str) -> str:

    value = os.environ.get(name, '').strip()

    if not value:

        raise RuntimeError(f'Missing {name}. Set it in your shell, the project-root .env file, or the local openai-api file.')

    return value



def run_node(js_code: str) -> dict:

    env = os.environ.copy()

    env['PATH'] = f"{NODE22.parent}:{env['PATH']}" if NODE22.exists() else env['PATH']

    env['NODE_PATH'] = str(NODE_PKG / 'node_modules')

    try:

        completed = subprocess.run(

            [NODE_BIN, '-e', js_code],

            cwd=NODE_PKG,

            env=env,

            capture_output=True,

            text=True,

            check=True,

        )

    except subprocess.CalledProcessError as exc:

        stdout = (exc.stdout or '').strip()

        stderr = (exc.stderr or '').strip()

        details = [f'Node command failed with exit code {exc.returncode}.']

        if stdout:

            details.append(f'STDOUT:\n{stdout}')

        if stderr:

            details.append(f'STDERR:\n{stderr}')

        raise RuntimeError('\n\n'.join(details)) from exc

    output = completed.stdout.strip()

    if not output:

        raise RuntimeError('Node command completed without JSON output.')

    return json.loads(output)



print('Node package path:', NODE_PKG)

print('Python executable:', sys.executable)

print('Node runtime:', NODE_BIN)

print('OPENAI_API_KEY loaded:', bool(os.environ.get('OPENAI_API_KEY')))


Node package path: /Users/hemantkumarthapa/Desktop/RAG/VB/embrix-node
Python executable: /Users/hemantkumarthapa/Desktop/RAG/VB/embrix-chain/backend/.venv/bin/python
Node runtime: /opt/homebrew/opt/node@22/bin/node
OPENAI_API_KEY loaded: True


In [2]:
# 1) Vector store + graph demo
js_code = textwrap.dedent('''
const { VectorStore, StateGraph, START, END } = require('./dist');

const store = new VectorStore({ dimension: 4 });
store.upsert('demo', [
  { id: 'a', values: [1, 0, 0, 0], text: 'agents and tools', metadata: { topic: 'agent' } },
  { id: 'b', values: [0, 1, 0, 0], text: 'graphs and workflows', metadata: { topic: 'graph' } },
  { id: 'c', values: [0, 0, 1, 0], text: 'retrieval and vectors', metadata: { topic: 'rag' } },
]);
const search = store.query('demo', [0, 0, 1, 0], 2);

const graph = new StateGraph();
graph.addNode('retrieve', async (state) => ({ ...state, hits: search.map((x) => x.text) }));
graph.addNode('respond', async (state) => ({ ...state, answer: `Top hit: ${state.hits[0]}` }));
graph.addEdge(START, 'retrieve');
graph.addEdge('retrieve', 'respond');
graph.addEdge('respond', END);

(async () => {
  const result = await graph.compile().invoke({ question: 'show retrieval' });
  console.log(JSON.stringify({ search, result }, null, 2));
})();
''')
run_node(js_code)

{'search': [{'id': 'c',
   'score': 1,
   'text': 'retrieval and vectors',
   'metadata': {'topic': 'rag'}},
  {'id': 'a',
   'score': 0,
   'text': 'agents and tools',
   'metadata': {'topic': 'agent'}}],
 'result': {'question': 'show retrieval',
  'hits': ['retrieval and vectors', 'agents and tools'],
  'answer': 'Top hit: retrieval and vectors'}}

In [6]:
# 2) Agent demo with the Node package and an OpenAI-compatible model

require_env('OPENAI_API_KEY')

js_code = textwrap.dedent(r'''

const { Agent, Tool, OpenAICompatibleChatModel } = require('./dist');



const model = new OpenAICompatibleChatModel({

  model: process.env.OPENAI_MODEL || 'gpt-4o-mini',

  apiKey: process.env.OPENAI_API_KEY,

  baseURL: process.env.OPENAI_BASE_URL || undefined,

  systemPrompt: 'You are a concise demo assistant.'

});



const calculator = new Tool('calculator', 'Evaluate arithmetic expressions', async (input) => {

  if (!/^[0-9+\-*/(). ]+$/.test(input)) throw new Error('Unsafe expression');

  return String(Function(`return (${input})`)());

});



(async () => {

  const agent = new Agent({ tools: [calculator], llm: model });

  const answer = await agent.run('Use the calculator tool to compute (44 / 2) + 9 and answer in one sentence.');

  console.log(JSON.stringify({ answer }, null, 2));

})();

''')

run_node(js_code)


{'answer': 'The result of (44 / 2) + 9 is 31.'}

In [8]:
# 3) RAGPipeline demo with a custom deterministic embedder

js_code = textwrap.dedent(r'''

const path = require('path');

const crypto = require('crypto');

const { BaseEmbedder, OpenAICompatibleChatModel, RAGPipeline, VectorStore } = require('./dist');



class HashEmbedder extends BaseEmbedder {

  constructor(dimension = 16) {

    super();

    this.dimension = dimension;

  }

  async embed(texts) {

    return texts.map((text) => {

      const vec = Array(this.dimension).fill(0);

      for (const token of String(text).toLowerCase().split(/\s+/).filter(Boolean)) {

        const digest = crypto.createHash('sha256').update(token).digest();

        const idx = digest[0] % this.dimension;

        vec[idx] += 1;

      }

      const norm = Math.sqrt(vec.reduce((acc, x) => acc + x * x, 0)) || 1;

      return vec.map((x) => x / norm);

    });

  }

}



const embedder = new HashEmbedder();

const model = new OpenAICompatibleChatModel({

  model: process.env.OPENAI_MODEL || 'gpt-4o-mini',

  apiKey: process.env.OPENAI_API_KEY,

  baseURL: process.env.OPENAI_BASE_URL || undefined,

  systemPrompt: 'Answer only from the provided context.'

});

const ragDbPath = path.join(process.cwd(), '..', 'demo_notebooks', 'embrix_node_rag.db');

const store = new VectorStore({ dimension: embedder.dimension, dbPath: ragDbPath });



(async () => {

  const rag = new RAGPipeline(embedder, model, store);

  await rag.ingest([

    'Embrix Node provides local vector search, graphs, agents, and DB tooling.',

    'The Node package is ideal for TypeScript apps, API servers, and developer tooling.',

    'RAG uses retrieval before generation to keep answers grounded.'

  ], 'knowledge');

  const answer = await rag.query('What makes the Node package useful?', 'knowledge');

  console.log(JSON.stringify({ answer }, null, 2));

})();

''')

run_node(js_code)


{'answer': 'The Node package is useful because it is ideal for TypeScript apps, API servers, and developer tooling.'}

In [9]:
# 4) Database vectorization + DBQueryEngine demo

js_code = textwrap.dedent(r'''

const fs = require('fs');

const path = require('path');

const crypto = require('crypto');

const BetterSQLite = require('better-sqlite3');

const {

  BaseEmbedder,

  DBQueryEngine,

  DBVectorizer,

  OpenAICompatibleChatModel,

  SQLiteConnector,

  VectorStore,

} = require('./dist');



class HashEmbedder extends BaseEmbedder {

  constructor(dimension = 16) {

    super();

    this.dimension = dimension;

  }

  async embed(texts) {

    return texts.map((text) => {

      const vec = Array(this.dimension).fill(0);

      for (const token of String(text).toLowerCase().split(/\s+/).filter(Boolean)) {

        const digest = crypto.createHash('sha256').update(token).digest();

        const idx = digest[0] % this.dimension;

        vec[idx] += 1;

      }

      const norm = Math.sqrt(vec.reduce((acc, x) => acc + x * x, 0)) || 1;

      return vec.map((x) => x / norm);

    });

  }

}



const dbPath = path.join(process.cwd(), '..', 'demo_notebooks', 'embrix_node_products.sqlite');

if (fs.existsSync(dbPath)) fs.unlinkSync(dbPath);

const seedDb = new BetterSQLite(dbPath);

seedDb.exec(`

  CREATE TABLE products (

    id INTEGER PRIMARY KEY,

    name TEXT,

    category TEXT,

    price REAL,

    description TEXT

  );

`);

const insert = seedDb.prepare('INSERT INTO products (name, category, price, description) VALUES (?, ?, ?, ?)');

insert.run('Trail Runner Pro', 'shoes', 89.0, 'lightweight running shoe for trails and long distance');

insert.run('Studio Comfort', 'headphones', 129.0, 'over ear headphones with clear sound and strong bass');

insert.run('City Sprint', 'shoes', 59.0, 'everyday running shoe for short road runs');

seedDb.close();



const connector = new SQLiteConnector(dbPath);

const embedder = new HashEmbedder();

const vectorDbPath = path.join(process.cwd(), '..', 'demo_notebooks', 'embrix_node_db_vectors.sqlite');

const store = new VectorStore({ dimension: embedder.dimension, dbPath: vectorDbPath });

const model = new OpenAICompatibleChatModel({

  model: process.env.OPENAI_MODEL || 'gpt-4o-mini',

  apiKey: process.env.OPENAI_API_KEY,

  baseURL: process.env.OPENAI_BASE_URL || undefined,

  systemPrompt: 'Return concise product recommendations.'

});



(async () => {

  const vectorizer = new DBVectorizer(connector, embedder, { store });

  const count = await vectorizer.vectorizeTable('products', 'products_demo', { textColumns: ['name', 'description', 'category'] });

  const engine = new DBQueryEngine(vectorizer, undefined, { connector, llm: model });

  const hits = await engine.search('budget running shoes', 'products_demo', 2);

  const answer = await engine.query('Which product fits a runner on a tighter budget?', 'products_demo', 2);

  const sqlRows = await engine.sqlQuery('Show product names and prices under 100 dollars', 'products');

  console.log(JSON.stringify({ count, hits, answer, sqlRows }, null, 2));

})();

''')

run_node(js_code)


{'count': 3,
 'hits': [{'id': 'products__3',
   'score': 0.6666666666666667,
   'text': 'name: City Sprint; description: everyday running shoe for short road runs; category: shoes',
   'metadata': {'id': 3,
    'name': 'City Sprint',
    'category': 'shoes',
    'price': 59,
    'description': 'everyday running shoe for short road runs',
    '__table': 'products'}},
  {'id': 'products__1',
   'score': 0.5360562677155593,
   'text': 'name: Trail Runner Pro; description: lightweight running shoe for trails and long distance; category: shoes',
   'metadata': {'id': 1,
    'name': 'Trail Runner Pro',
    'category': 'shoes',
    'price': 89,
    'description': 'lightweight running shoe for trails and long distance',
    '__table': 'products'}}],
 'answer': 'Trail Runner Pro',
 'sqlRows': [{'name': 'Trail Runner Pro', 'price': 89},
  {'name': 'City Sprint', 'price': 59}]}